# Rumore e ritorno: come funziona la diffusione

Il codice del capitolo [«Rumore e ritorno: come funziona la diffusione»](https://book.paithon.it/main/ModelliDiffusione/come-funziona.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Rumore e ritorno: come funziona la diffusione

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/come-funziona.html)


### La diffusione in miniatura: una spirale di punti


In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)
rng = np.random.default_rng(0)

# --- Dati: 2000 punti disposti a spirale, coordinate in [-1, 1] ---
n = 2000
angolo = 3.0 * np.pi * np.sqrt(rng.uniform(size=n))    # angolo lungo la spirale
raggio = angolo / (3.0 * np.pi)                        # il raggio cresce con l'angolo
spirale = np.stack([raggio * np.cos(angolo),
                    raggio * np.sin(angolo)], axis=1)  # shape (2000, 2)
spirale += 0.02 * rng.standard_normal(spirale.shape)   # leggero spessore del tratto
x0 = torch.tensor(spirale, dtype=torch.float32)        # (2000, 2)

# --- Schedule del rumore: lo stesso di DDPM ---
T = 1000
beta = torch.linspace(1e-4, 0.02, T)       # beta_t, shape (T,)
alpha = 1.0 - beta                         # alpha_t
alpha_bar = torch.cumprod(alpha, dim=0)    # alpha_t barrato, shape (T,)

def rumorizza(x0, t, eps):
    """Forma chiusa dell'andata: x_t dato x_0, per t interi in [0, T-1]."""
    ab = alpha_bar[t].unsqueeze(1)                     # (B, 1)
    return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps    # (B, 2)

In [ ]:
def embedding_tempo(t, dim=16):
    """Embedding sinusoidale del passo t: da (B,) a (B, dim)."""
    freq = torch.exp(torch.arange(dim // 2) * (-np.log(10000.0) / (dim // 2)))
    ang = t.float().unsqueeze(1) * freq.unsqueeze(0)   # (B, dim/2)
    return torch.cat([ang.sin(), ang.cos()], dim=1)    # (B, dim)

class PredittoreRumore(nn.Module):
    """La rete epsilon_theta(x_t, t): un MLP al posto della U-Net."""
    def __init__(self, dim_t=16, dim_h=128):
        super().__init__()
        self.dim_t = dim_t
        self.rete = nn.Sequential(
            nn.Linear(2 + dim_t, dim_h), nn.SiLU(),
            nn.Linear(dim_h, dim_h), nn.SiLU(),
            nn.Linear(dim_h, 2),                       # stima del rumore 2D
        )

    def forward(self, x, t):
        emb = embedding_tempo(t, self.dim_t)           # (B, dim_t)
        return self.rete(torch.cat([x, emb], dim=1))   # (B, 2)

In [ ]:
modello = PredittoreRumore()
ottimizzatore = torch.optim.Adam(modello.parameters(), lr=2e-3)

for passo in range(30000):
    idx = torch.randint(0, n, (256,))         # minibatch di 256 punti
    batch = x0[idx]                           # (256, 2)
    t = torch.randint(0, T, (256,))           # un livello di rumore per esempio
    eps = torch.randn_like(batch)             # il rumore "vero" (la soluzione)
    x_t = rumorizza(batch, t, eps)            # (256, 2)
    predetto = modello(x_t, t)                # (256, 2), rumore stimato
    loss = ((eps - predetto) ** 2).mean()     # MSE: la loss semplice di DDPM
    ottimizzatore.zero_grad()
    loss.backward()
    ottimizzatore.step()
    if passo % 5000 == 0:
        print(f"passo {passo:5d}  loss {loss.item():.3f}")

In [ ]:
@torch.no_grad()
def campiona(n_campioni=1000):
    """Percorre la catena inversa da x_T (rumore puro) a x_0."""
    x = torch.randn(n_campioni, 2)                        # x_T ~ N(0, I)
    for t in reversed(range(T)):
        t_batch = torch.full((n_campioni,), t)            # (B,), tutti uguali a t
        eps_pred = modello(x, t_batch)                    # rumore stimato
        coeff = beta[t] / (1.0 - alpha_bar[t]).sqrt()     # quanto se ne toglie
        # mossa 1 (correggi) e mossa 2 (riscala), in una riga: mu_theta(x_t, t)
        media = (x - coeff * eps_pred) / alpha[t].sqrt()
        if t > 0:
            # mossa 3 (rimescola): sigma_t * z, con sigma_t = sqrt(beta_t).
            # E' il termine piu' grande dei tre, non un ritocco
            x = media + beta[t].sqrt() * torch.randn_like(x)
        else:
            x = media                    # ultimo passo: niente rumore fresco
    return x                             # (n_campioni, 2)

nuovi = campiona()
print(nuovi.shape)   # torch.Size([1000, 2]): mille coppie di coordinate

# "nuovi" e' una parola grossa: verifichiamola senza disegnare niente. Si
# confronta quanto dista un punto generato dal piu' vicino dell'archivio con
# quanto distano fra loro due punti dell'archivio: se i due numeri si
# somigliano, i generati cadono *fra* quelli di partenza, cioe' sulla spirale
# ma in posti dove non c'era nessuno
da_archivio = torch.cdist(nuovi, x0).min(dim=1).values
fra_archivio = torch.cdist(x0, x0).fill_diagonal_(float("inf")).min(dim=1).values
print(f"generato -> archivio: {da_archivio.median():.3f}   "
      f"archivio -> archivio: {fra_archivio.median():.3f}")
# generato -> archivio: 0.009   archivio -> archivio: 0.006

## Il limite continuo: una sola equazione, in avanti e all'indietro

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/sde-e-ode.html)


### Le due famiglie: chi lascia esplodere la varianza e chi la conserva


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# programma di rumore VP, lo stesso di DDPM in versione continua
B_MIN, B_MAX, T = 0.1, 20.0, 1.0
beta = lambda t: B_MIN + t * (B_MAX - B_MIN)
alpha = lambda t: np.exp(-0.5 * (B_MIN * t + 0.5 * (B_MAX - B_MIN) * t**2))
sigma = lambda t: np.sqrt(1 - alpha(t)**2)

# dati: due sole possibilita', cosi' tutto il resto e' calcolabile a mano
MODI = np.array([-1.5, 1.5])
x0 = rng.choice(MODI, size=20000)

# 1) l'equazione simulata a piccoli passi
N = 1000
dt = T / N
x = x0.copy()
fotografie = {}
for k in range(N):
    t = k * dt
    x = (x - 0.5 * beta(t) * x * dt
         + np.sqrt(beta(t) * dt) * rng.normal(size=x.shape))
    if k + 1 in (N // 4, N // 2, N):
        fotografie[(k + 1) * dt] = x.copy()

# 2) la stessa cosa in forma chiusa, senza simulare niente
print("  t     simulata          forma chiusa")
for t, xs in fotografie.items():
    chiusa = alpha(t) * x0 + sigma(t) * rng.normal(size=x0.shape)
    print(f"{t:5.2f}   {xs.mean():+.3f} {xs.std():.3f}     "
          f"{chiusa.mean():+.3f} {chiusa.std():.3f}")
# ->  0.25   +0.002 1.282     -0.001 1.286
# ->  0.50   -0.005 1.048     -0.015 1.051
# ->  1.00   +0.005 0.999     +0.008 0.995

### La strada deterministica, quella che non sorteggia niente


In [ ]:
def punteggio(x, t):
    """Il punteggio esatto della mistura, senza nessuna rete."""
    a, s = alpha(t), sigma(t)
    d = x[:, None] - a * MODI[None, :]
    w = np.exp(-0.5 * (d / s)**2)
    w /= w.sum(axis=1, keepdims=True)
    return -(w * d).sum(axis=1) / s**2

n_camp, M = 5000, 500
dt = T / M

# la strada deterministica: dal rumore ai dati, senza sorteggiare niente
y = rng.normal(size=n_camp)
for k in range(M):
    t = T - k * dt
    y = y + dt * 0.5 * beta(t) * (y + punteggio(y, t))

# la strada casuale: la stessa cosa con il tremore acceso
z = rng.normal(size=n_camp)
for k in range(M):
    t = T - k * dt
    z = (z + dt * (0.5 * beta(t) * z + beta(t) * punteggio(z, t))
         + np.sqrt(beta(t) * dt) * rng.normal(size=n_camp))

for nome, v in (("ODE", y), ("SDE", z)):
    print(nome, round(float((v < 0).mean()), 4),
          round(float(v[v < 0].mean()), 4), round(float(v[v > 0].mean()), 4),
          round(float(np.abs(np.abs(v) - 1.5).mean()), 4))
# -> ODE 0.501 -1.5003 1.5 0.0055
# -> SDE 0.4998 -1.5001 1.5005 0.0135

In [ ]:
ts = np.linspace(1e-3, T, M + 1)
campo = lambda x, t: -0.5 * beta(t) * (x + punteggio(x, t))

partenza = rng.choice(MODI, size=6) + 0.01 * rng.normal(size=6)
x = partenza.copy()
for k in range(M):                      # dai dati al rumore
    x = x + (ts[k + 1] - ts[k]) * campo(x, ts[k])
rumore = x.copy()
for k in range(M, 0, -1):               # e ritorno, sulla stessa strada
    x = x - (ts[k] - ts[k - 1]) * campo(x, ts[k])

print(np.round(rumore, 3))
# -> [-0.354  2.779  0.572 -0.04   0.562 -0.623]
print(round(float(np.abs(x - partenza).max()), 4))   # -> 0.0077

### In pratica


In [ ]:
# la conversione fra le quattro parametrizzazioni, che nessuna libreria
# nasconde ma tutte chiamano in modo diverso
def converti(eps, x_t, t):
    a, s = alpha(t), sigma(t)
    x0 = (x_t - s * eps) / a
    score = -eps / s
    v = a * eps - s * x0
    return x0, score, v

t = 0.5
x_t = alpha(t) * MODI[0] + sigma(t) * 0.3        # un esempio costruito a mano
eps_vero = 0.3
x0, score, v = converti(eps_vero, x_t, t)
print(round(float(x0), 6), round(float(MODI[0]), 6))     # -> -1.5 -1.5
print(round(float(v), 6),
      round(float(alpha(t) * eps_vero - sigma(t) * MODI[0]), 6))
# -> 1.523836 1.523836

## Flow matching: scegliere la strada invece di ereditarla

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/flow-matching.html)


### Scegliere il percorso, e il più semplice è una retta


In [ ]:
import numpy as np

MODI = np.array([-1.5, 1.5])
T_MIN = 1e-3

# --- percorso rettificato: x_t = (1-t) x0 + t z, quindi x_t|x0 ha
#     media (1-t) x0 e deviazione t
def x0_atteso(x, t):
    d = x[:, None] - (1 - t) * MODI[None, :]
    w = np.exp(-0.5 * (d / t)**2)
    w /= w.sum(axis=1, keepdims=True)
    return (w * MODI[None, :]).sum(axis=1)

def velocita_retta(x, t):
    return (x - x0_atteso(x, t)) / t

# --- percorso della diffusione (VP), cioe' il campo della PF-ODE
B_MIN, B_MAX = 0.1, 20.0
beta = lambda t: B_MIN + t * (B_MAX - B_MIN)
alpha = lambda t: np.exp(-0.5 * (B_MIN * t + 0.5 * (B_MAX - B_MIN) * t**2))
sigma = lambda t: np.sqrt(1 - alpha(t)**2)

def velocita_diffusione(x, t):
    a, s = alpha(t), sigma(t)
    d = x[:, None] - a * MODI[None, :]
    w = np.exp(-0.5 * (d / s)**2)
    w /= w.sum(axis=1, keepdims=True)
    punteggio = -(w * d).sum(axis=1) / s**2
    return -0.5 * beta(t) * (x + punteggio)

rng = np.random.default_rng(0)
z = rng.normal(size=4000)

def integra(campo, passi):
    ts = np.linspace(1.0, T_MIN, passi + 1)
    x = z.copy()
    for k in range(passi):
        x = x + (ts[k + 1] - ts[k]) * campo(x, ts[k])
    return x

errore = lambda x: float(np.abs(np.abs(x) - 1.5).mean())

print("passi   retta    diffusione")
for passi in (1, 2, 4, 8, 16, 64, 256):
    print(f"{passi:5d}   {errore(integra(velocita_retta, passi)):.4f}   "
          f"{errore(integra(velocita_diffusione, passi)):.4f}")
# -> passi   retta    diffusione
# ->     1   1.4992   0.8112
# ->     2   0.4845   0.7442
# ->     4   0.0227   0.4300
# ->     8   0.0020   0.1204
# ->    16   0.0017   0.0343
# ->    64   0.0016   0.0135
# ->   256   0.0015   0.0104

In [ ]:
griglia = np.array([-3.0, -1.5, -0.15, 0.15, 1.5, 3.0])
arrivo = griglia.copy()
ts = np.linspace(1.0, T_MIN, 401)
for k in range(400):
    arrivo = arrivo + (ts[k + 1] - ts[k]) * velocita_retta(arrivo, ts[k])
print(np.round(arrivo, 4))            # -> [-1.5013 -1.4996 -1.4973  1.4973  1.4996  1.5013]
print(bool(np.all(np.diff(arrivo) > 0)))          # la mappa e' monotona -> True

## Lo spazio latente: Stable Diffusion

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/stable-diffusion.html)


### Dieci righe di Python


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
from diffusers import StableDiffusionPipeline

# carica l'intera pipeline (VAE + U-Net + CLIP + campionatore)
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,  # mezza precisione: meno memoria
)
pipe = pipe.to("cuda")          # sposta tutto sulla GPU

immagine = pipe(
    prompt="a black cat jumping on a wall, watercolor",
    negative_prompt="blurry, deformed, watermark",
    guidance_scale=7.5,         # il peso w della guidance
    num_inference_steps=50,     # i passi di denoising nel latente
).images[0]

immagine.save("gatto_acquerello.png")
```


## Quando la diffusione incontra i Transformer: DiT

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/diffusion-transformer.html)


### Un DiT in miniatura


In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(0)

def embedding_tempo(t, dim=128):
    """Embedding sinusoidale del passo t: da (B,) a (B, dim)."""
    freq = torch.exp(-math.log(10000.0) * torch.arange(dim // 2) / (dim // 2))
    ang = t.float().unsqueeze(1) * freq.unsqueeze(0)     # (B, dim/2)
    return torch.cat([ang.sin(), ang.cos()], dim=1)      # (B, dim)

class BloccoDiT(nn.Module):
    """Attenzione + MLP, con modulazione adaLN-zero dal condizionamento."""
    def __init__(self, d=128, teste=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)  # LN "nuda"
        self.attn = nn.MultiheadAttention(d, teste, batch_first=True)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(),
                                 nn.Linear(4 * d, d))
        # dal condizionamento: shift, scale e gate per i due sotto-strati
        self.manopole = nn.Linear(d, 6 * d)
        nn.init.zeros_(self.manopole.weight)   # adaLN-ZERO: blocco = identita'
        nn.init.zeros_(self.manopole.bias)

    def forward(self, x, c):
        # x: (B, N, d) token del latente; c: (B, d) tempo + classe
        b1, g1, a1, b2, g2, a2 = self.manopole(c).chunk(6, dim=1)  # 6 x (B, d)
        h = self.norm1(x) * (1 + g1.unsqueeze(1)) + b1.unsqueeze(1)
        att, _ = self.attn(h, h, h, need_weights=False)
        x = x + a1.unsqueeze(1) * att            # gate a1: vale 0 all'inizio
        h = self.norm2(x) * (1 + g2.unsqueeze(1)) + b2.unsqueeze(1)
        x = x + a2.unsqueeze(1) * self.mlp(h)    # gate a2, idem
        return x

class MiniDiT(nn.Module):
    """DiT minimale: patchify, blocchi Transformer, patch di rumore in uscita."""
    def __init__(self, canali=4, lato=32, patch=2, d=128, blocchi=4, classi=10):
        super().__init__()
        self.canali, self.lato, self.patch = canali, lato, patch
        n_token = (lato // patch) ** 2                       # (32/2)^2 = 256
        self.patchify = nn.Conv2d(canali, d, kernel_size=patch, stride=patch)
        self.pos = nn.Parameter(torch.zeros(1, n_token, d))  # posizioni apprese
        self.emb_classe = nn.Embedding(classi, d)
        self.blocchi = nn.ModuleList([BloccoDiT(d) for _ in range(blocchi)])
        self.finale = nn.Linear(d, patch * patch * canali)   # token -> sua patch

    def forward(self, z, t, y):
        # z: (B, 4, 32, 32) latente rumoroso; t: (B,) passo; y: (B,) classe
        x = self.patchify(z)                     # (B, d, 16, 16)
        x = x.flatten(2).transpose(1, 2)         # (B, 256, d): i token
        x = x + self.pos
        c = embedding_tempo(t, x.shape[-1]) + self.emb_classe(y)   # (B, d)
        for blocco in self.blocchi:
            x = blocco(x, c)
        x = self.finale(x)                       # (B, 256, patch*patch*4)
        # ricompone le patch: l'uscita ha la stessa forma dell'ingresso
        B, g, p, C = z.shape[0], self.lato // self.patch, self.patch, self.canali
        x = x.view(B, g, g, p, p, C)             # (B, 16, 16, 2, 2, 4)
        x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, self.lato, self.lato)
        return x                                 # (B, 4, 32, 32): rumore stimato

modello = MiniDiT()
z = torch.randn(2, 4, 32, 32)      # due latenti fittizi, come quelli del VAE
t = torch.randint(0, 1000, (2,))   # un passo di rumore per ciascuno
y = torch.randint(0, 10, (2,))     # una classe per ciascuno
print(modello(z, t, y).shape)      # torch.Size([2, 4, 32, 32])
print(sum(p.numel() for p in modello.parameters()))  # 1225616: ~1.2 milioni

# la verifica di adaLN-zero, che le due righe sopra non fanno: a pesi
# appena inizializzati ogni blocco deve essere l'identita', per qualunque c
blocco, x, c = modello.blocchi[0], torch.randn(2, 256, 128), torch.randn(2, 128)
print((blocco(x, c) - x).abs().max().item())   # 0.0

## Meno passi: risolvere l'equazione invece di simularla

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/campionatori-veloci.html)


### DDIM è il primo gradino della scala, e si vede


In [ ]:
import numpy as np

MODI = np.array([-1.5, 1.5])
T_MIN = 1e-3
B_MIN, B_MAX = 0.1, 20.0
beta = lambda t: B_MIN + t * (B_MAX - B_MIN)
alpha = lambda t: np.exp(-0.5 * (B_MIN * t + 0.5 * (B_MAX - B_MIN) * t**2))
sigma = lambda t: np.sqrt(1 - alpha(t)**2)
lam = lambda t: np.log(alpha(t) / sigma(t))          # log-SNR

def eps(x, t):
    """La rete perfetta: il rumore atteso, in forma chiusa."""
    a, s = alpha(t), sigma(t)
    d = x[:, None] - a * MODI[None, :]
    w = np.exp(-0.5 * (d / s)**2)
    w /= w.sum(axis=1, keepdims=True)
    return (w * d).sum(axis=1) / s

# t in funzione di lambda, invertito una volta sola su una griglia fitta
_t = np.linspace(T_MIN, 1.0, 200_001)
_l = lam(_t)
t_di_lam = lambda L: float(np.interp(L, _l[::-1], _t[::-1]))

rng = np.random.default_rng(0)
z = rng.normal(size=2000)

def eulero(M):
    """Eulero in t: tratta l'equazione come una scatola nera."""
    ts = np.linspace(1.0, T_MIN, M + 1)
    x = z.copy()
    for k in range(M):
        t = ts[k]
        campo = -0.5 * beta(t) * x + 0.5 * beta(t) / sigma(t) * eps(x, t)
        x = x + (ts[k + 1] - t) * campo
    return x

def ddim(M):
    """Integratore esponenziale del primo ordine, a passi uguali in lambda."""
    Ls = np.linspace(lam(1.0), lam(T_MIN), M + 1)
    x = z.copy()
    for k in range(M):
        s, t = t_di_lam(Ls[k]), t_di_lam(Ls[k + 1])
        h = Ls[k + 1] - Ls[k]
        x = alpha(t) / alpha(s) * x - sigma(t) * (np.exp(h) - 1) * eps(x, s)
    return x

def dpm2(M):
    """DPM-Solver del secondo ordine: due valutazioni per passo."""
    Ls = np.linspace(lam(1.0), lam(T_MIN), M + 1)
    x = z.copy()
    for k in range(M):
        s, t = t_di_lam(Ls[k]), t_di_lam(Ls[k + 1])
        h = Ls[k + 1] - Ls[k]
        m = t_di_lam(Ls[k] + h / 2)
        u = alpha(m) / alpha(s) * x - sigma(m) * (np.exp(h / 2) - 1) * eps(x, s)
        x = alpha(t) / alpha(s) * x - sigma(t) * (np.exp(h) - 1) * eps(u, m)
    return x

riferimento = dpm2(1000)                     # la soluzione, praticamente esatta
scarto = lambda x: float(np.abs(x - riferimento).mean())

print("valutazioni   Eulero      DDIM        DPM-2")
misure = {}
for N in (8, 16, 32, 64, 128):
    misure[N] = (scarto(eulero(N)), scarto(ddim(N)), scarto(dpm2(N // 2)))
    print(f"{N:11d}   {misure[N][0]:.3e}   {misure[N][1]:.3e}   "
          f"{misure[N][2]:.3e}")
# -> valutazioni   Eulero      DDIM        DPM-2
# ->           8   1.164e-01   1.265e-02   5.940e-02
# ->          16   3.015e-02   4.737e-03   5.012e-03
# ->          32   1.059e-02   2.063e-03   6.528e-04
# ->          64   5.673e-03   9.624e-04   1.411e-04
# ->         128   3.593e-03   4.650e-04   3.655e-05

print("ordine misurato:", [round(float(np.log(misure[32][i] / misure[128][i])
                                       / np.log(4)), 2) for i in range(3)])
# -> ordine misurato: [0.78, 1.07, 2.08]

### In pratica: quale campionatore


In [ ]:
# la stessa qualita' con quattro budget di valutazioni della rete
for N in (8, 16, 32, 64):
    print(N, f"{misure[N][1]:.1e}", f"{misure[N][2]:.1e}",
          "primo ordine" if misure[N][1] < misure[N][2] else "secondo ordine")
# -> 8 1.3e-02 5.9e-02 primo ordine
# -> 16 4.7e-03 5.0e-03 primo ordine
# -> 32 2.1e-03 6.5e-04 secondo ordine
# -> 64 9.6e-04 1.4e-04 secondo ordine

## Da mille passi a uno: insegnare a saltare

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/pochi-passi.html)


### Il quadro che tiene insieme tutto: imparare la mappa, non la velocità


In [ ]:
import numpy as np

MODI = np.array([-1.5, 1.5])
T_MIN = 1e-3
B_MIN, B_MAX = 0.1, 20.0
beta = lambda t: B_MIN + t * (B_MAX - B_MIN)
alpha = lambda t: np.exp(-0.5 * (B_MIN * t + 0.5 * (B_MAX - B_MIN) * t**2))
sigma = lambda t: np.sqrt(1 - alpha(t)**2)

def eps(x, t):
    a, s = alpha(t), sigma(t)
    d = x[:, None] - a * MODI[None, :]
    w = np.exp(-0.5 * (d / s)**2)
    w /= w.sum(axis=1, keepdims=True)
    return (w * d).sum(axis=1) / s

campo = lambda x, t: -0.5 * beta(t) * x + 0.5 * beta(t) / sigma(t) * eps(x, t)

def percorri(x, da, a, passi=400):
    ts = np.linspace(da, a, passi + 1)
    for k in range(passi):
        x = x + (ts[k + 1] - ts[k]) * campo(x, ts[k])
    return x

rng = np.random.default_rng(0)
z = rng.normal(size=8)

destinazione = percorri(z.copy(), 1.0, T_MIN)     # f(z, 1): dove si va a finire
print(np.round(destinazione, 4))
# -> [ 1.4842 -1.4845  1.4995  1.4829 -1.4972  1.493   1.5109  1.5051]

# autoconsistenza: fermarsi lungo la strada e ripartire da li'
for t_meta in (0.6, 0.3, 0.1):
    a_meta = percorri(z.copy(), 1.0, t_meta)
    scarto = np.abs(percorri(a_meta, t_meta, T_MIN) - destinazione).max()
    print(t_meta, round(float(scarto), 6))
# -> 0.6 0.000888
# -> 0.3 0.001641
# -> 0.1 0.002068

In [ ]:
velocita_media = (destinazione - z) / (T_MIN - 1.0)
print(np.round(z + (T_MIN - 1.0) * velocita_media, 4))
# -> [ 1.4842 -1.4845  1.4995  1.4829 -1.4972  1.493   1.5109  1.5051]

print(np.round(z + (T_MIN - 1.0) * campo(z, 1.0), 4))
# -> [ 0.1258 -0.1322  0.6408  0.105  -0.536   0.3618  1.3047  0.9476]

## Piegare la generazione: guida, vincoli, preferenze

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/guida.html)


### Che cosa la guida fa davvero alla distribuzione


In [ ]:
import numpy as np

MU, S, T_MIN = np.array([-1.5, 1.5]), 0.5, 1e-3
B_MIN, B_MAX = 0.1, 20.0
beta = lambda t: B_MIN + t * (B_MAX - B_MIN)
alpha = lambda t: np.exp(-0.5 * (B_MIN * t + 0.5 * (B_MAX - B_MIN) * t**2))

def score_condizionato(x, t, k):
    """Con la classe nota i dati sono una gaussiana sola: punteggio esatto."""
    a = alpha(t)
    v = a * a * S * S + (1 - a * a)
    return -(x - a * MU[k]) / v

def score_libero(x, t):
    a = alpha(t)
    v = a * a * S * S + (1 - a * a)
    d = x[:, None] - a * MU[None, :]
    w = np.exp(-0.5 * d * d / v)
    w /= w.sum(axis=1, keepdims=True)
    return -(w * d).sum(axis=1) / v

def genera(forza, n=20000, passi=400):
    rng = np.random.default_rng(0)
    x = rng.normal(size=n)
    ts = np.linspace(1.0, T_MIN, passi + 1)
    for i in range(passi):
        t, dt = ts[i], ts[i + 1] - ts[i]
        libero = score_libero(x, t)
        guidato = libero + forza * (score_condizionato(x, t, 1) - libero)
        x = x + dt * (-0.5 * beta(t) * x - 0.5 * beta(t) * guidato)
    return x

# la distribuzione che la lettura corrente attribuisce alla guida
xg = np.linspace(-8, 8, 400_001)
gauss = lambda m: np.exp(-0.5 * ((xg - m) / S)**2)
p = 0.5 * gauss(MU[0]) + 0.5 * gauss(MU[1])
p_classe = gauss(MU[1]) / (gauss(MU[0]) + gauss(MU[1]))

print("forza |  campionatore vero  |  p(x) p(c|x)^w")
for forza in (1.0, 3.0, 7.5):
    x = genera(forza)
    q = p * p_classe**forza
    q /= np.trapezoid(q, xg)
    m = np.trapezoid(xg * q, xg)
    s = np.sqrt(np.trapezoid((xg - m)**2 * q, xg))
    print(f"{forza:5.1f} |  {x.mean():+.3f}  {x.std():.3f}      |  "
          f"{m:+.3f}  {s:.3f}")
# -> forza |  campionatore vero  |  p(x) p(c|x)^w
# ->   1.0 |  +1.499  0.498      |  +1.500  0.500
# ->   3.0 |  +2.019  0.311      |  +1.505  0.494
# ->   7.5 |  +2.570  0.247      |  +1.508  0.490

### In quale tratto del percorso nasce lo spostamento


In [ ]:
def genera_a_intervallo(forza, da, a, n=20000, passi=400):
    rng = np.random.default_rng(0)
    x = rng.normal(size=n)
    ts = np.linspace(1.0, T_MIN, passi + 1)
    divari = []
    for i in range(passi):
        t, dt = ts[i], ts[i + 1] - ts[i]
        libero = score_libero(x, t)
        divario = score_condizionato(x, t, 1) - libero
        # fuori dalla finestra la forza torna a uno, cioe' nessuna spinta
        # in piu' di quella che il teorema di Bayes autorizza
        qui = forza if da <= t <= a else 1.0
        if qui != 1.0:
            divari.append(np.abs(divario).mean())
        x = x + dt * (-0.5 * beta(t) * x
                      - 0.5 * beta(t) * (libero + qui * divario))
    return x, np.mean(divari)

def riga(nome, ris):
    x, divario = ris
    print(f"{nome:22} {x.mean():+.3f}   {x.std():.3f}    "
          f"{x.mean() - 1.5:+.3f}   {divario:.3f}")

print(f"{'':22} media  larghezza  scarto  divario")
print(f"{'bersaglio':22} +1.500   0.500")
riga("guida sempre accesa", genera_a_intervallo(7.5, 0.0, 1.0))
for da, a in ((0.8, 1.0), (0.6, 0.8), (0.4, 0.6), (0.2, 0.4), (0.0, 0.2)):
    riga(f"accesa fra {da} e {a}", genera_a_intervallo(7.5, da, a))
# ->                        media  larghezza  scarto  divario
# -> bersaglio              +1.500   0.500
# -> guida sempre accesa    +2.570   0.247    +1.070   0.052
# -> accesa fra 0.8 e 1.0   +1.661   0.492    +0.161   0.028
# -> accesa fra 0.6 e 0.8   +2.030   0.417    +0.530   0.119
# -> accesa fra 0.4 e 0.6   +2.199   0.232    +0.699   0.203
# -> accesa fra 0.2 e 0.4   +1.857   0.240    +0.357   0.137
# -> accesa fra 0.0 e 0.2   +1.546   0.427    +0.046   0.027

## Quando lo stato è fatto di simboli: diffondere il testo

[Leggi la pagina](https://book.paithon.it/main/ModelliDiffusione/diffusione-discreta.html)


### Il prezzo del parallelismo


In [ ]:
import itertools
from collections import Counter
import numpy as np

# Un linguaggio di quattro parole: tre bit, e il numero di uni deve essere
# pari. Ogni bit da solo e' cinquanta e cinquanta, ogni coppia di bit e'
# indipendente, ma i tre insieme sono legatissimi: due qualsiasi decidono
# il terzo.
L = 3
LINGUA = [w for w in itertools.product((0, 1), repeat=L) if sum(w) % 2 == 0]
print(LINGUA)                # -> [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)]

MASCHERA = -1

def probabilita_di_uno(stato, i):
    """p(bit i = 1 | quello che si e' gia' scoperto), per enumerazione."""
    compatibili = [w for w in LINGUA
                   if all(s == MASCHERA or w[j] == s
                          for j, s in enumerate(stato))]
    return sum(1 for w in compatibili if w[i] == 1) / len(compatibili)

def campiona(passi, n=20000):
    rng = np.random.default_rng(0)
    fuori = []
    for _ in range(n):
        stato = [MASCHERA] * L
        ordine = list(rng.permutation(L))
        # le posizioni si scoprono a gruppi: `passi` gruppi in tutto
        quanti = [L // passi + (1 if k < L % passi else 0) for k in range(passi)]
        for q in quanti:
            gruppo = [ordine.pop() for _ in range(q)]
            # tutte insieme, ciascuna dalla propria probabilita' condizionata
            scelte = {i: int(rng.random() < probabilita_di_uno(stato, i))
                      for i in gruppo}
            for i, v in scelte.items():
                stato[i] = v
        fuori.append(tuple(stato))
    return fuori

for passi in (1, 2, 3):
    conta = Counter(campiona(passi))
    n = sum(conta.values())
    valide = sum(v for w, v in conta.items() if sum(w) % 2 == 0) / n
    freq = sorted(v / n for v in conta.values())
    print(f"passi {passi}: valide {valide:.3f}, risultati diversi"
          f" {len(conta)}, dal {freq[0]:.3f} al {freq[-1]:.3f}")
# -> passi 1: valide 0.499, risultati diversi 8, dal 0.123 al 0.129
# -> passi 2: valide 1.000, risultati diversi 4, dal 0.249 al 0.252
# -> passi 3: valide 1.000, risultati diversi 4, dal 0.249 al 0.252